# ConstructIA — Avance de proyecto 2 | Componente de PLN
**Materia:** Desarrollo de Aplicaciones con Inteligencia Artificial — Tecmilenio.

Este cuaderno da continuidad al chatbot basado en reglas del Avance 1. Para el Avance 2 construimos un **dataset sintético de mensajes** etiquetados con las **mismas 15 intenciones**, y evaluamos un modelo de clasificación mediante TF-IDF + Naive Bayes/SVM.

> **Importante:** todos los mensajes del dataset son ejemplos creados para el proyecto, no conversaciones de clientes reales. El desempeño medido corresponde únicamente a estos datos de prueba. La CNN y la integración texto-imagen se realizarán en la siguiente etapa.

## 1. Preparar Google Colab y cargar los archivos del repositorio
Ejecuta esta celda una vez. Usa **Entorno de ejecución → Ejecutar todas**. No necesitas descargar archivos manualmente.

In [ ]:
%pip -q install nltk pandas scikit-learn matplotlib
import os, sys, subprocess
repo = '/content/ConstructIA'
if os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git', '-C', repo, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/chrisavm24-dot/ConstructIA.git', repo], check=True)
if os.path.join(repo, 'avance2') not in sys.path:
    sys.path.insert(0, os.path.join(repo, 'avance2'))
import nltk, sklearn, pandas as pd
print('Python:', sys.version.split()[0], '| NLTK:', nltk.__version__, '| scikit-learn:', sklearn.__version__)
assert sys.version_info >= (3, 10)

## 2. Dataset de intenciones
**180 mensajes, 15 intenciones, 12 ejemplos por categoría.** El archivo CSV incluye la columna `origen` para hacer explícito que son ejemplos sintéticos. Primero verificamos que no haya mensajes duplicados.

In [ ]:
from nlp_modelo import cargar_datos, obtener_stopwords, procesar_texto, entrenar_y_evaluar, predecir
datos = cargar_datos()
print('Mensajes:', len(datos), '| Intenciones:', datos['intencion'].nunique())
print('Distribución por intención:')
display(datos.groupby('intencion').size().rename('mensajes').to_frame())
display(datos.sample(8, random_state=3)[['texto','intencion','origen']])

## 3. Cuatro técnicas de preprocesamiento
- **Normalización**: minúsculas, limpieza de espacios y tildes.
- **Tokenización**: separación en palabras con NLTK.
- **Eliminación de stopwords**: lista explícita de palabras frecuentes en español; conservamos *no*, *sin*, *nunca* y *ni*.
- **Stemming**: reducción a raíces con `SnowballStemmer('spanish')` de NLTK.

Se utiliza **stemming**, una de las dos alternativas admitidas en la rúbrica (*lematización o stemming*).

In [ ]:
ejemplos = [
    'Hola, necesito rentar una retroexcavadora en Monterrey',
    '¿La plataforma viene sin operador?',
    'Quiero cotizar una remodelación en Guadalupe',
]
for e in ejemplos:
    p = procesar_texto(e)
    print('
ORIGINAL:', p['original'])
    print('NORMALIZACIÓN:', p['normalizado'])
    print('TOKENS:', p['tokens'])
    print('SIN STOPWORDS:', p['sin_stopwords'])
    print('STEMMING:', p['stemming'])

## 4. TF-IDF, entrenamiento y selección del clasificador
Separamos **75% para entrenamiento** y **25% para prueba** de manera estratificada. **Dentro del entrenamiento** comparamos Naive Bayes y SVM con validación cruzada en cuatro partes, usando **macro F1**. Después se entrena el mejor y se evalúa una sola vez con mensajes reservados para prueba.

El vectorizador TF-IDF se ajusta únicamente con el conjunto de entrenamiento para evitar filtración de información del conjunto de prueba.

In [ ]:
resultado = entrenar_y_evaluar()
print('Conjunto:',len(resultado['datos']),'mensajes; entrenamiento:',resultado['metricas']['num_entrenamiento'],'| prueba:',resultado['metricas']['num_prueba'])
print('Representación: TF-IDF, vocabulario de',resultado['metricas']['vocabulario_tfidf'],'términos')
print('Comparación F1 macro en validación cruzada (solo entrenamiento):',resultado['metricas']['f1_cv_entrenamiento'])
print('Modelo elegido:',resultado['metricas']['modelo_elegido'])
print('Accuracy en prueba:',round(resultado['metricas']['accuracy'],3))
print('Precision macro:',round(resultado['metricas']['precision_macro'],3))
print('Recall macro:',round(resultado['metricas']['recall_macro'],3))
print('F1 macro:',round(resultado['metricas']['f1_macro'],3))

## 5. Evaluación del modelo textual
Aquí se muestran el informe por clase, la matriz de confusión y los ejemplos que clasificó incorrectamente. **Un resultado bajo no es un error de ejecución**: revela que necesitamos más mensajes variados y resolver ambigüedades entre intenciones.

In [ ]:
print(resultado['reporte'])
print('Errores observados en mensajes no usados para entrenar:')
display(resultado['detalles'].loc[~resultado['detalles']['correcto']])

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(figsize=(12, 10))
ConfusionMatrixDisplay(confusion_matrix=resultado['matriz'], display_labels=resultado['etiquetas']).plot(ax=ax, xticks_rotation=90, colorbar=False)
ax.set_title('ConstructIA: matriz de confusión — datos sintéticos reservados para prueba')
plt.tight_layout()
plt.show()

## 6. Consultas adicionales de demostración
El modelo se ejecuta con frases nuevas. **No inventamos porcentajes de confianza**: SVM lineal no entrega probabilidades calibradas directamente. Para uso comercial, necesitaríamos un mecanismo confiable de detección de consultas fuera de alcance y revisión humana.

In [ ]:
muestras = [
 'Necesito una excavadora para mañana',
 'Quiero saber el precio de una plataforma elevadora',
 '¿Pueden llevar una máquina a Santiago?',
 'Me gustaría hablar con alguien de ventas',
 'No quiero operador para mi equipo',
]
for texto in muestras:
    print(predecir(resultado['pipeline'],texto))

## 7. Interpretación y continuidad multimodal
**Qué se logró:** dataset etiquetado, cuatro técnicas de preprocesamiento, representación TF-IDF, comparación Naive Bayes/SVM, evaluación con accuracy, precision, recall y F1, más matriz de confusión.

**Limitaciones:** dataset pequeño y sintético; solo 12 ejemplos por clase. Se espera confusión entre categorías relacionadas (por ejemplo, *zonas* y *disponibilidad*). Los resultados no son generalizables a clientes reales.

**Próxima fase:** preparar fotografías etiquetadas de **retroexcavadora**, **plataforma elevadora** y **minicargador** con derechos de uso documentados; separar entrenamiento, validación y prueba sin duplicados; entrenar una **ResNet50 con transferencia de aprendizaje**; reportar accuracy, precision, recall, F1 y matriz de confusión; integrar predicción visual con una explicación en el chatbot.

**Evidencias para el Word:** captura de dataset, ejemplo de preprocesamiento, tabla de métricas, matriz de confusión y errores observados. Adjuntar enlace a este cuaderno en GitHub, no presentar el dataset como datos reales de una empresa.